# Imports and config

In [1]:
# ====================== INSTALL REQUIREMENTS ======================
!pip install ultralytics roboflow
!pip install supervision==0.26.0
!pip install SoccerNet
!pip install kornia
!pip install opencv-python
!pip install segmentation-models-pytorch

import ultralytics
import torch

ultralytics.checks()

Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6960.8/8062.4 GB disk)


In [2]:
# ====================== IMPORTS ======================
# Standard Library
import shutil
import argparse
import base64
import colorsys
import glob
import json
import os
import pickle
import random
import re
import subprocess
import sys
import tempfile
from argparse import Namespace
from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import partial
from io import BytesIO
from multiprocessing import Pool
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Third-party: Data & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv

# Third-party: Image Processing
import cv2
from cv2 import pointPolygonTest
from PIL import Image
import torchvision.transforms as T

# Third-party: Machine Learning
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from scipy.ndimage import median_filter
from scipy.signal import savgol_filter

# Third-party: Deep Learning
import torch
from transformers import AutoProcessor, SiglipVisionModel
from ultralytics import YOLO

# Third-party: Utilities
from kaggle_secrets import UserSecretsClient
from more_itertools import chunked
from tqdm import tqdm
from tqdm.auto import tqdm as tqdm_auto

In [3]:
# ====================== CONFIGURATION ======================
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 12

SEED = 33
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Utils and Helpers

In [4]:
# ====================== VIDEO WRITER ======================

def save_frames_to_video(
    frames: List[np.ndarray],
    output_path: Union[str, Path],
    fps: float,
    codec: str = "mp4v",
) -> str:
    """
    Save a list of BGR frames to a video file.

    Args:
        frames: List of OpenCV frames in BGR format
        output_path: Path to the output video file (e.g. .mp4)
        fps: Frames per second of the output video
        codec: FourCC codec code (default: "mp4v")

    Returns:
        str: Absolute path to the saved video file

    Raises:
        ValueError: If the frames list is empty
        RuntimeError: If VideoWriter cannot be opened
    """
    if len(frames) == 0:
        raise ValueError("Frame list is empty.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Get dimensions from the first frame
    h, w = frames[0].shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (w, h),
    )

    if not writer.isOpened():
        raise RuntimeError(f"Failed to open VideoWriter: {output_path}")

    for frame in frames:
        # Ensure all frames have the same size
        if frame.shape[:2] != (h, w):
            frame = cv2.resize(frame, (w, h))

        writer.write(frame)

    writer.release()

    return str(output_path)

In [5]:
# ====================== HELPERS: FRAME INDEX & VIDEO METADATA ======================

def frame_index_from_image_id(image_id: str) -> int:
    """
    Extract frame number from image filename.

    Expected format: 'frame_000123.png'

    Args:
        image_id: Filename of the frame image

    Returns:
        int: Frame index

    Raises:
        ValueError: If frame number cannot be parsed
    """
    match = re.search(r"frame_(\d+)", str(image_id))
    if match is None:
        raise ValueError(f"Could not extract frame index from image_id: {image_id}")
    
    return int(match.group(1))


def get_video_fps(video_path: Union[str, Path]) -> float:
    """
    Get FPS (frames per second) of a video file.

    Args:
        video_path: Path to the video file

    Returns:
        float: FPS value
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    return fps


def read_first_n_frames(
    video_path: Union[str, Path],
    n: int = 32,
) -> Tuple[List[np.ndarray], float]:
    """
    Read the first N frames from a video file.

    Useful for quick previews or testing.

    Args:
        video_path: Path to the video
        n: Number of frames to read

    Returns:
        Tuple[List[np.ndarray], float]: 
            - List of frames in BGR format
            - Video FPS
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frames: List[np.ndarray] = []

    while len(frames) < n:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        frames.append(frame_bgr)

    cap.release()
    return frames, fps

In [6]:
# ====================== VIDEO FRAME BATCHING ======================

def iter_video_frame_batches(
    video_path: Union[str, Path],
    batch_size: int = 128,
    start_frame: int = 0,
    max_frames: Optional[int] = None,
):
    """
    Iterate through video and yield batches of frames.

    Yields:
        Tuple[List[int], List[np.ndarray]]: 
            - frame_indices: list of frame numbers
            - frames_bgr: list of frames in BGR format (OpenCV default)
    """
    video_path = str(video_path)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frame_indices: List[int] = []
    frames_bgr: List[np.ndarray] = []
    current_idx = start_frame
    read_count = 0

    while True:
        if max_frames is not None and read_count >= max_frames:
            break

        ret, frame = cap.read()
        if not ret:
            break

        frame_indices.append(current_idx)
        frames_bgr.append(frame)

        current_idx += 1
        read_count += 1

        if len(frames_bgr) >= batch_size:
            yield frame_indices, frames_bgr
            frame_indices = []
            frames_bgr = []

    # Yield remaining frames (last incomplete batch)
    if frames_bgr:
        yield frame_indices, frames_bgr

    cap.release()


def save_frame_batch_to_disk(
    frames_bgr: List[np.ndarray],
    frame_indices: List[int],
    batch_dir: Union[str, Path],
) -> List[str]:
    """
    Save a batch of frames to disk.

    Frames are saved with consistent naming: frame_000123.png
    This format is important for downstream compatibility with
    ViewTransformer and homography mapping.
    """
    batch_dir = Path(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)

    image_ids: List[str] = []

    for frame, frame_idx in zip(frames_bgr, frame_indices):
        image_id = f"frame_{frame_idx:06d}.png"
        image_path = batch_dir / image_id

        ok = cv2.imwrite(str(image_path), frame)
        if not ok:
            raise RuntimeError(f"Failed to save frame: {image_path}")

        image_ids.append(image_id)

    return image_ids


def prepare_temp_root(
    temp_root: Union[str, Path], 
    clean: bool = True
) -> Path:
    """
    Prepare temporary directory for storing frame batches.

    Args:
        temp_root: Path to the temporary directory
        clean: If True, remove existing directory and its contents

    Returns:
        Path: Prepared directory path
    """
    temp_root = Path(temp_root)

    if clean and temp_root.exists():
        shutil.rmtree(temp_root)

    temp_root.mkdir(parents=True, exist_ok=True)
    return temp_root

In [7]:
# ====================== BBOX UTILS & KINEMATICS ======================

def get_center_of_bbox(bbox: list) -> tuple[int, int]:
    """Return the center coordinates of a bounding box."""
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def get_bbox_width(bbox: list) -> int:
    """Return the width of a bounding box."""
    return int(bbox[2] - bbox[0])


def measure_distance(p1: tuple, p2: tuple) -> float:
    """Calculate Euclidean distance between two points."""
    return ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5


def measure_xy_distance(p1: tuple, p2: tuple) -> tuple[float, float]:
    """Calculate dx, dy between two points."""
    return p1[0] - p2[0], p1[1] - p2[1]


def get_foot_position(bbox: list) -> list[int]:
    """Return the position of the player's foot (bottom center of bbox)."""
    x1, y1, x2, y2 = bbox
    return [int((x1 + x2) / 2), int(y2)]


def get_closest_to_point_bbox(bbox1: list, bbox2: list, point: tuple) -> list:
    """Return the bounding box closer to the given point."""
    if measure_distance(get_center_of_bbox(bbox1), point) < measure_distance(
        get_center_of_bbox(bbox2), point
    ):
        return bbox1
    return bbox2


def calculate_kinematics(tracks: dict, fps: int = 25) -> dict:
    """
    Calculate smoothed speed for players, goalkeepers and referees.

    Uses median filter + Savitzky-Golay smoothing to remove noise
    and unrealistic speed spikes caused by homography jitter.
    """
    categories = ["players", "goalkeepers", "referees"]

    for cat in categories:
        if cat not in tracks:
            continue

        trajectories = {}

        # Collect trajectories per track_id
        for frame_idx, frame_data in enumerate(tracks[cat]):
            for track_id, info in frame_data.items():
                if "position_field" not in info:
                    continue

                if track_id not in trajectories:
                    trajectories[track_id] = {"frames": [], "coords": []}

                trajectories[track_id]["frames"].append(frame_idx)
                trajectories[track_id]["coords"].append(info["position_field"])

        # Process each trajectory
        for track_id, data in trajectories.items():
            coords = np.array(data["coords"])
            N = len(coords)

            if N < 7:  # Need enough points for reliable smoothing
                continue

            # 1. Median filter removes sudden jumps ("teleports")
            coords_med_x = median_filter(coords[:, 0], size=3)
            coords_med_y = median_filter(coords[:, 1], size=3)

            # 2. Savitzky-Golay smoothing
            window = min(15, N if N % 2 == 1 else N - 1)
            smooth_x = savgol_filter(coords_med_x, window, polyorder=2)
            smooth_y = savgol_filter(coords_med_y, window, polyorder=2)

            # 3. Calculate distance and speed (km/h)
            dx = np.diff(smooth_x)
            dy = np.diff(smooth_y)
            dist = np.sqrt(dx**2 + dy**2)

            speed_kmh = dist * fps * 3.6  # m/frame * fps * 3.6 = km/h

            # 4. Smooth the speed itself
            if len(speed_kmh) > 5:
                speed_window = min(9, len(speed_kmh) // 2 * 2 - 1)
                speed_kmh = savgol_filter(speed_kmh, speed_window, polyorder=1)

            # Write results back to tracks
            frames = data["frames"]
            for i in range(len(frames)):
                # Speed for current frame is based on movement from previous frame
                s = float(speed_kmh[i - 1]) if i > 0 else float(speed_kmh[0])

                # Cap at realistic maximum (protection against homography errors)
                tracks[cat][frames[i]][track_id]["speed"] = min(s, 40.0)

    return tracks

In [8]:
# ====================== VIDEO UTILS ======================

import os
import cv2
from tqdm import tqdm


def load_video_frames(video_path: str):
    """Return a generator that yields frames from the video."""
    return sv.get_video_frames_generator(video_path)


def process_tracks(
    frame_generator, 
    tracks: dict, 
    team_classifier, 
    ball_control: bool = False
):
    """
    Process tracks: assign teams and optionally track ball control.
    """
    if ball_control:
        player_assigner = PlayerBallAssigner()
        team_ball_control = []

        for frame_id, frame in enumerate(frame_generator):
            players = tracks["players"][frame_id]
            goalkeepers = tracks["goalkeepers"][frame_id]

            team_classifier.process_frame(frame, players, goalkeepers)

            # Assign ball to closest player
            ball_bbox = tracks['ball'][frame_id][0]['bbox']
            assigned_player = player_assigner.assign_ball_to_player(players, ball_bbox)

            if assigned_player != -1:
                tracks['players'][frame_id][assigned_player]['has_ball'] = True
                try:
                    team_ball_control.append(
                        tracks['players'][frame_id][assigned_player]['team_id']
                    )
                except:
                    continue
            elif len(team_ball_control) > 0:
                team_ball_control.append(team_ball_control[-1])

        return np.array(team_ball_control)
    else:
        # Only team classification
        for frame_id, frame in enumerate(frame_generator):
            players = tracks["players"][frame_id]
            goalkeepers = tracks["goalkeepers"][frame_id]
            team_classifier.process_frame(frame, players, goalkeepers)

        return None


def render_video(frame_generator, tracks: dict, team_ball_control=None):
    """Generate annotated frames with tracks and optional ball control."""
    return draw_annotations(frame_generator, tracks, team_ball_control)


def save_video(frames, output_path: str, fps: int = 30) -> None:
    """
    Save list/generator of frames to video file with fallback codecs.
    """
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
        print(f"Created directory: {output_dir}")

    writer = None
    frame_count = 0
    h, w = 0, 0

    try:
        for frame in frames:
            if frame is None:
                continue

            # Initialize writer on first valid frame
            if writer is None:
                h, w = frame.shape[:2]
                
                # Try primary codec
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

                # Fallback to XVID if mp4v fails
                if not writer.isOpened():
                    print("[WARNING] mp4v not supported, trying XVID...")
                    fourcc = cv2.VideoWriter_fourcc(*"XVID")
                    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

                if not writer.isOpened():
                    raise RuntimeError("Failed to initialize VideoWriter")

                print(f"Video created: {output_path}")
                print(f"Resolution: {w}x{h}, FPS: {fps}")

            # Ensure consistent frame size
            if frame.shape[1] != w or frame.shape[0] != h:
                frame = cv2.resize(frame, (w, h))

            writer.write(frame)
            frame_count += 1

        if writer is not None:
            writer.release()

        if frame_count == 0:
            print("[ERROR] No frames were provided")
        else:
            print(f"Video successfully saved: {output_path} ({frame_count} frames)")

    except Exception as e:
        print(f"[ERROR] Failed to save video: {e}")
        if writer is not None:
            writer.release()


def build_radar_video(
    tracks: dict, 
    mapper, 
    output_path: str, 
    fps: int = 25
) -> None:
    """
    Create a radar (minimap) video from tracking data.

    Args:
        tracks: Dictionary with tracking data
        mapper: Object with get_blank_template() and to_pixels(x, y) methods
        output_path: Output video path
        fps: Frames per second
    """
    total_frames = len(tracks["players"])
    if total_frames == 0:
        print("No frames to render")
        return

    # Get radar frame size
    template = mapper.get_blank_template()
    h, w = template.shape[:2]

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    # Fallback codec
    if not out.isOpened():
        print("mp4v failed, trying XVID...")
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    for frame_idx in tqdm(range(total_frames), desc="Rendering radar video"):
        radar_frame = draw_basic_radar(frame_idx, tracks, mapper)
        out.write(radar_frame)

    out.release()
    print(f"Radar video saved: {output_path}")

In [9]:
# ====================== ANNOTATION UTILS ======================

def draw_ellipse(
    frame: np.ndarray, 
    bbox: list, 
    color: tuple, 
    track_id: Optional[int] = None
) -> np.ndarray:
    """
    Draw an ellipse under the object (typically for players).
    """
    x_center, y_bottom = get_center_of_bbox(bbox)[0], int(bbox[3])
    width = get_bbox_width(bbox)

    cv2.ellipse(
        frame,
        center=(x_center, y_bottom),
        axes=(width, int(0.35 * width)),
        angle=0.0,
        startAngle=-45,
        endAngle=235,
        color=color,
        thickness=2,
        lineType=cv2.LINE_4,
    )

    if track_id is not None:
        # Background rectangle for track_id
        rect_x1 = x_center - 20
        rect_y1 = y_bottom + 5
        rect_x2 = x_center + 20
        rect_y2 = y_bottom + 25

        cv2.rectangle(frame, (rect_x1, rect_y1), (rect_x2, rect_y2), color, cv2.FILLED)

        # Adjust text position for 3-digit numbers
        text_x = rect_x1 + (2 if track_id > 99 else 12)
        cv2.putText(
            frame,
            str(track_id),
            (text_x, rect_y1 + 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 0),
            2,
        )
    return frame


def draw_triangle(
    frame: np.ndarray, 
    bbox: list, 
    color: tuple, 
    track_id: Optional[int] = None
) -> np.ndarray:
    """
    Draw a triangle above the object (used for ball possession).
    """
    x_center, y_top = get_center_of_bbox(bbox)[0], int(bbox[1])

    triangle_points = np.array([
        [x_center, y_top],
        [x_center - 10, y_top - 20],
        [x_center + 10, y_top - 20]
    ], np.int32)

    cv2.drawContours(frame, [triangle_points], 0, color, cv2.FILLED)
    cv2.drawContours(frame, [triangle_points], 0, (0, 0, 0), 2)

    if track_id is not None and track_id != "":
        text_x = x_center - 15
        text_y = y_top - 25

        # Background for better text visibility
        (text_w, text_h), _ = cv2.getTextSize(
            str(track_id), cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
        )
        cv2.rectangle(
            frame,
            (text_x - 2, text_y - text_h - 2),
            (text_x + text_w + 2, text_y + 2),
            (0, 0, 0),
            cv2.FILLED,
        )

        cv2.putText(
            frame,
            str(track_id),
            (text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2,
            cv2.LINE_AA,
        )
    return frame


def draw_annotations(
    frame_generator, 
    tracks: dict, 
    team_ball_control=None
):
    """
    Generator that adds visual annotations (ellipses, triangles, etc.) to frames.
    """
    for frame_num, frame in enumerate(frame_generator):
        frame = frame.copy()

        players = tracks["players"][frame_num]
        referees = tracks["referees"][frame_num]
        balls = tracks["ball"][frame_num]

        # Draw players
        for track_id, player in players.items():
            color = player.get("team_color", (0, 0, 255))  # red by default
            frame = draw_ellipse(frame, player["bbox"], color, track_id)

            if player.get("has_ball", False):
                frame = draw_triangle(frame, player["bbox"], (0, 0, 255))

        # Draw referees
        for ref in referees.values():
            frame = draw_ellipse(frame, ref["bbox"], (0, 255, 255))  # cyan

        # Draw ball
        for ball in balls.values():
            frame = draw_triangle(frame, ball["bbox"], (0, 255, 0))

        # Draw ball possession statistics
        if team_ball_control is not None:
            frame = draw_team_ball_control(frame, frame_num, team_ball_control)

        yield frame


def draw_basic_radar(frame_idx: int, tracks: dict, mapper) -> np.ndarray:
    """
    Draw a basic radar (minimap) view with players, goalkeepers, referees and ball.
    """
    radar = mapper.get_blank_template()

    # Players & Goalkeepers
    for cat in ["players", "goalkeepers"]:
        for tid, info in tracks[cat][frame_idx].items():
            if "position_field" not in info:
                continue

            x_m, y_m = info["position_field"]
            px, py = mapper.to_pixels(x_m, y_m)

            color = info.get("team_color", (255, 255, 255))

            cv2.circle(radar, (px, py), 8, color, -1)
            cv2.circle(radar, (px, py), 9, (255, 255, 255), 1)  # white border

            if cat == "goalkeepers":
                cv2.putText(
                    radar, "G", (px - 4, py + 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1
                )

    # Referees
    for ref_info in tracks["referees"][frame_idx].values():
        if "position_field" not in ref_info:
            continue

        x_m, y_m = ref_info["position_field"]
        px, py = mapper.to_pixels(x_m, y_m)

        referee_color = (0, 128, 255)  # orange
        cv2.circle(radar, (px, py), 7, referee_color, -1)
        cv2.circle(radar, (px, py), 8, (255, 255, 255), 1)
        cv2.putText(
            radar, "R", (px - 3, py + 3),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1
        )

    # Ball
    if tracks["ball"][frame_idx]:
        ball_data = tracks["ball"][frame_idx].get(0)
        if ball_data and "position_field" in ball_data:
            bx, by = mapper.to_pixels(*ball_data["position_field"])
            cv2.circle(radar, (bx, by), 5, (255, 255, 255), -1)
            cv2.circle(radar, (bx, by), 6, (0, 0, 0), 1)

    return radar


def draw_speed(frames, tracks: dict) -> list:
    """
    Draw player speed (km/h) on frames.
    """
    output_frames = []

    for frame_num, frame in enumerate(frames):
        frame = frame.copy()

        for obj_type in ["players", "goalkeepers"]:
            if frame_num >= len(tracks[obj_type]):
                continue

            for track_info in tracks[obj_type][frame_num].values():
                if "speed" not in track_info:
                    continue

                speed_kmh = track_info.get("speed", 0.0)
                bbox = track_info["bbox"]
                foot_x, foot_y = get_foot_position(bbox)

                cv2.putText(
                    frame,
                    f"{speed_kmh:.1f} km/h",
                    (foot_x - 30, foot_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 0, 0),
                    2,
                    cv2.LINE_AA,
                )

        output_frames.append(frame)

    return output_frames


def draw_team_ball_control(
    frame: np.ndarray, 
    frame_num: int, 
    team_ball_control: np.ndarray
) -> np.ndarray:
    """
    Draw team ball possession statistics in the bottom-right corner.
    """
    if team_ball_control is None:
        return frame

    overlay = frame.copy()
    cv2.rectangle(overlay, (1350, 850), (1900, 970), (255, 255, 255), -1)
    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)

    control_till_now = team_ball_control[:frame_num + 1]
    if len(control_till_now) == 0:
        return frame

    t1 = np.sum(control_till_now == 0) / len(control_till_now)
    t2 = np.sum(control_till_now == 1) / len(control_till_now)

    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(
        frame, 
        f"Team 1 Ball Control: {t1*100:.2f}%", 
        (1400, 900), font, 1, (0, 0, 0), 3
    )
    cv2.putText(
        frame, 
        f"Team 2 Ball Control: {t2*100:.2f}%", 
        (1400, 950), font, 1, (0, 0, 0), 3
    )

    return frame


def interpolate_ball_positions(ball_tracks: list) -> list:
    """
    Interpolate missing or noisy ball positions using linear interpolation.
    """
    ball_positions = []
    for frame_data in ball_tracks:
        if frame_data:
            first_key = list(frame_data.keys())[0]
            bbox = frame_data[first_key]['bbox']
            ball_positions.append({
                'x1': bbox[0], 'y1': bbox[1],
                'x2': bbox[2], 'y2': bbox[3]
            })
        else:
            ball_positions.append({
                'x1': None, 'y1': None, 'x2': None, 'y2': None
            })

    df_ball = pd.DataFrame(ball_positions)

    # Calculate center and remove big jumps
    df_ball['mid_x'] = (df_ball['x1'] + df_ball['x2']) / 2
    df_ball['mid_y'] = (df_ball['y1'] + df_ball['y2']) / 2

    dist = np.sqrt(df_ball['mid_x'].diff()**2 + df_ball['mid_y'].diff()**2)
    df_ball.loc[dist > 150, ['x1', 'y1', 'x2', 'y2']] = np.nan

    # Interpolate
    df_ball = df_ball.interpolate(method='linear', limit_direction='both', limit=30)
    df_ball = df_ball.bfill()

    # Convert back to original format
    interpolated = []
    for _, row in df_ball.iterrows():
        interpolated.append({
            0: {"bbox": [row['x1'], row['y1'], row['x2'], row['y2']]}
        })

    return interpolated

# Calibration module

In [10]:
# ====================== CLONE TVCALIB REPOSITORY ======================
print("=== Cloning tvcalib with sn_segmentation submodule ===")

TVCAIB_REPO = "https://github.com/MM4SPA/tvcalib"
TARGET_DIR = "tvcalib"
SN_SEGMENTATION_DIR = os.path.join(TARGET_DIR, "sn_segmentation")
SOURCE_REPO = "https://github.com/jtheiner/sn-calibration-segmentation"

# Clone tvcalib if it doesn't exist
if not os.path.exists(TARGET_DIR):
    print(f"Cloning {TVCAIB_REPO}...")
    subprocess.run(["git", "clone", TVCAIB_REPO, TARGET_DIR], check=True)
else:
    print(f"Directory {TARGET_DIR} already exists, updating...")
    subprocess.run(["git", "-C", TARGET_DIR, "pull"], check=True)

# Remove old sn_segmentation directory if it exists
if os.path.exists(SN_SEGMENTATION_DIR):
    print(f"Removing old {SN_SEGMENTATION_DIR} directory...")
    shutil.rmtree(SN_SEGMENTATION_DIR)

# Clone sn-calibration-segmentation into the correct location
print(f"Cloning {SOURCE_REPO} into {SN_SEGMENTATION_DIR}...")
subprocess.run(["git", "clone", SOURCE_REPO, SN_SEGMENTATION_DIR], check=True)

# Verify the structure
if os.path.exists(os.path.join(SN_SEGMENTATION_DIR, "src", "segmentation")):
    print("✅ Success! sn_segmentation directory has been replaced with sn-calibration-segmentation content")
else:
    print("❌ Error: folder structure does not match expected")

# Remove .git directory inside sn_segmentation to avoid conflicts
git_dir = os.path.join(SN_SEGMENTATION_DIR, ".git")
if os.path.exists(git_dir):
    shutil.rmtree(git_dir)
    print("🗑️ Removed .git from sn_segmentation (keeping tvcalib as the main repository)")

# Change working directory and set up Python path
%cd /kaggle/working/tvcalib

sys.path.insert(0, "/kaggle/working/tvcalib")
sys.path.insert(0, "/kaggle/working/tvcalib/sn_segmentation")

print("=== Installing tvcalib package ===")
!pip install -e .

print("=== Installing compatible dependencies (Kaggle) ===")
!pip install kornia==0.6.3 pytorch-lightning==1.5.10 soccernet==0.1.34 \
    matplotlib==3.5.1 pandas tqdm opencv-python-headless

# ====================== DOWNLOAD SEGMENTATION MODEL ======================
print("=== Downloading pretrained segmentation model ===")
!mkdir -p data/segment_localization
!wget -q --show-progress https://tib.eu/cloud/s/x68XnTcZmsY4Jpg/download/train_59.pt -O data/segment_localization/train_59.pt
print("✅ Everything is ready!")

# ====================== PATCHES ======================
BASE = "/kaggle/working/tvcalib/tvcalib"


def replace_in_file(path: str, old: str, new: str) -> None:
    """Replace text in a file."""
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    text = text.replace(old, new)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


# PATCH 1: torch._six removal
file_path = f"{BASE}/sncalib_dataset.py"
replace_in_file(
    file_path,
    "from torch._six import string_classes",
    "import collections.abc\nstring_classes = str",
)
print("✅ torch._six patched")

# Additional safety patches
files_to_patch = [f"{BASE}/sncalib_dataset.py"]

for fp in files_to_patch:
    with open(fp, "r", encoding="utf-8") as f:
        text = f.read()

    # Replace deprecated Iterable import
    text = re.sub(
        r"from collections import Iterable",
        "from collections.abc import Iterable",
        text,
    )

    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)

print("✅ collections patches applied")

=== Cloning tvcalib with sn_segmentation submodule ===
Cloning https://github.com/MM4SPA/tvcalib...


Cloning into 'tvcalib'...


Removing old tvcalib/sn_segmentation directory...
Cloning https://github.com/jtheiner/sn-calibration-segmentation into tvcalib/sn_segmentation...


Cloning into 'tvcalib/sn_segmentation'...


✅ Success! sn_segmentation directory has been replaced with sn-calibration-segmentation content
🗑️ Removed .git from sn_segmentation (keeping tvcalib as the main repository)
/kaggle/working/tvcalib
=== Installing tvcalib package ===
Obtaining file:///kaggle/working/tvcalib
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editable ... error
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
=== Installing compatible dependencies (Kaggle) ===
Requested pytorch-lightning==1.5.10 from http

In [11]:
# ====================== TVCalib OPTIMIZED BATCH PIPELINE ======================

from SoccerNet.Evaluation.utils_calibration import SoccerPitch
from tvcalib.module import TVCalibModule
from tvcalib.cam_distr.tv_main_center import get_cam_distr, get_dist_distr
from tvcalib.sncalib_dataset import custom_list_collate
from tvcalib.utils.io import detach_dict, tensor2list
from tvcalib.utils.objects_3d import (
    SoccerPitchLineCircleSegments,
    SoccerPitchSNCircleCentralSplit,
)
from tvcalib.inference import (
    InferenceDatasetSegmentation,
    InferenceDatasetCalibration,
    InferenceSegmentationModel,
    get_camera_from_per_sample_output,
)
from tvcalib.utils import visualization_mpl_min as viz
from sn_segmentation.src.custom_extremities import (
    generate_class_synthesis,
    get_line_extremities,
)


# ====================== HELPER FUNCTIONS ======================
def get_valid_video_path() -> str:
    """Prompt user for video path and validate it."""
    while True:
        path = input("\nEnter the full path to the video file: ").strip()
        
        # Remove quotes if user accidentally copied with them
        path = path.strip('"\'')
        
        if not path:
            print("❌ Path cannot be empty. Please try again.")
            continue
            
        if not os.path.exists(path):
            print(f"❌ File not found: {path}")
            print("Please check the path and try again.\n")
            continue
            
        if not os.path.isfile(path):
            print(f"❌ Path exists but is not a file: {path}")
            continue
            
        # Optional: check common video extensions
        valid_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv'}
        ext = os.path.splitext(path)[1].lower()
        if ext not in valid_extensions:
            print(f"⚠️  Warning: File extension '{ext}' is not a common video format.")
            confirm = input("Continue anyway? (y/n): ").strip().lower()
            if confirm != 'y':
                continue
        
        print(f"✅ Video path accepted: {path}")
        return path


# ====================== CONFIGURATION ======================
@dataclass
class TVCalibConfig:
    """Configuration for TVCalib batch processing pipeline."""
    
    # Video path will be set interactively
    video_path: str
    
    # Temporary directory for batch frames
    temp_root: str = "/kaggle/working/tvcalib_temp_batches"
    
    # Path to TVCalib segmentation model checkpoint
    checkpoint: str = "data/segment_localization/train_59.pt"
    
    # Original video dimensions
    image_width: int = 1920
    image_height: int = 1080
    
    # Test limitation
    max_frames: Optional[int] = 750
    
    # Number of frames saved to disk per outer batch
    frame_batch_size: int = 128
    
    # Batch sizes for neural network processing
    batch_size_seg: int = 8
    batch_size_calib: int = 64
    
    # Number of workers for DataLoader and post-processing
    nworkers: int = 4
    
    # TVCalib optimization parameters
    optim_steps: int = 3000
    lens_dist: bool = False
    gpu: bool = True
    
    # Keypoint extraction parameters from segmentation
    line_skeleton_radius: int = 4
    extremities_maxdist: int = 30
    extremities_width: int = 455
    extremities_height: int = 256
    num_points_lines: int = 4
    num_points_circles: int = 8

class TVCalibContext:
    """
    Container for models, field objects and helper functions.
    Used to avoid reinitializing TVCalib for every batch.
    """
    def __init__(self, config: TVCalibConfig):
        self.config = config
        self.device = "cuda" if config.gpu and torch.cuda.is_available() else "cpu"
        print(f"TVCalib device: {self.device}")

        # Allow safe deserialization of argparse.Namespace
        torch.serialization.add_safe_globals([Namespace])

        self.object3d = SoccerPitchLineCircleSegments(
            device=self.device,
            base_field=SoccerPitchSNCircleCentralSplit(),
        )
        
        self.object3dcpu = SoccerPitchLineCircleSegments(
            device="cpu",
            base_field=SoccerPitchSNCircleCentralSplit(),
        )

        self.fn_generate_class_synthesis = partial(
            generate_class_synthesis,
            radius=config.line_skeleton_radius,
        )

        self.fn_get_line_extremities = partial(
            get_line_extremities,
            maxdist=config.extremities_maxdist,
            width=config.extremities_width,
            height=config.extremities_height,
            num_points_lines=config.num_points_lines,
            num_points_circles=config.num_points_circles,
        )

        self.model_seg = InferenceSegmentationModel(
            config.checkpoint,
            self.device,
        )

        self.model_calib = TVCalibModule(
            self.object3d,
            get_cam_distr(1.96, config.batch_size_calib, 1),
            get_dist_distr(config.batch_size_calib, 1) if config.lens_dist else None,
            (config.image_height, config.image_width),
            config.optim_steps,
            self.device,
            log_per_step=False,
        )

# ====================== INITIALIZATION WITH USER INPUT ======================
def create_config() -> TVCalibConfig:
    """Create config with user-provided video path."""
    video_path = get_valid_video_path()
    
    return TVCalibConfig(
        video_path=video_path
    )

Seed set to 10


In [12]:
# ====================== SEGMENTATION ======================

def run_segmentation_on_image_dir(
    images_path: Union[str, Path],
    ctx: TVCalibContext,
) -> Tuple[List[str], List[dict]]:
    """
    Run TVCalib segmentation on all images in a directory.

    Args:
        images_path: Path to directory containing images
        ctx: TVCalibContext containing segmentation model and helpers

    Returns:
        Tuple[List[str], List[dict]]:
            - image_ids: List of image filenames
            - keypoints_raw: List of raw keypoint dictionaries (lines + circles)
    """
    config = ctx.config
    images_path = Path(images_path)

    dataset_seg = InferenceDatasetSegmentation(
        images_path,
        config.image_width,
        config.image_height,
    )

    if len(dataset_seg) == 0:
        return [], []

    dataloader_seg = torch.utils.data.DataLoader(
        dataset_seg,
        batch_size=config.batch_size_seg,
        num_workers=config.nworkers,
        shuffle=False,
        collate_fn=custom_list_collate,
    )

    image_ids: List[str] = []
    keypoints_raw: List[dict] = []

    for batch_dict in dataloader_seg:
        with torch.no_grad():
            sem_lines = ctx.model_seg.inference(
                batch_dict["image"].to(ctx.device)
            )

        # Convert to numpy for post-processing
        sem_lines = sem_lines.cpu().numpy().astype(np.uint8)

        # Parallel post-processing using multiprocessing Pool
        with Pool(config.nworkers) as pool:
            skeletons_batch = pool.map(
                ctx.fn_generate_class_synthesis,
                sem_lines,
            )
            keypoints_raw_batch = pool.map(
                ctx.fn_get_line_extremities,
                skeletons_batch,
            )

        image_ids.extend(batch_dict["image_id"])
        keypoints_raw.extend(keypoints_raw_batch)

    return image_ids, keypoints_raw

In [13]:
# ====================== CALIBRATION ======================

def run_calibration_from_keypoints(
    image_ids: List[str],
    keypoints_raw: List[dict],
    ctx: TVCalibContext,
) -> pd.DataFrame:
    """
    Run TVCalib camera calibration using extracted keypoints.

    Args:
        image_ids: List of image filenames
        keypoints_raw: List of raw keypoint dictionaries from segmentation
        ctx: TVCalibContext containing calibration model and configuration

    Returns:
        pd.DataFrame: DataFrame with calibration results (camera parameters,
                      losses, selected points, etc.) indexed by image_id
    """
    config = ctx.config

    if len(image_ids) == 0:
        return pd.DataFrame()

    dataset_calib = InferenceDatasetCalibration(
        keypoints_raw,
        config.image_width,
        config.image_height,
        ctx.object3d,
    )

    dataloader_calib = torch.utils.data.DataLoader(
        dataset_calib,
        batch_size=config.batch_size_calib,
        collate_fn=custom_list_collate,
    )

    # Collect results for all samples
    per_sample_output = defaultdict(list)
    per_sample_output["image_id"] = [[image_id] for image_id in image_ids]

    for x_dict in dataloader_calib:
        batch_size_current = x_dict["lines__ndc_projected_selection_shuffled"].shape[0]

        # Perform self-optimization (calibration)
        per_sample_loss, cam, _ = ctx.model_calib.self_optim_batch(x_dict)

        # Prepare output: camera parameters + losses
        output_dict = tensor2list(
            detach_dict({
                **cam.get_parameters(batch_size_current),
                **per_sample_loss,
            })
        )

        # Add projected points for visualization
        output_dict["points_line"] = x_dict["lines__px_projected_selection_shuffled"]
        output_dict["points_circle"] = x_dict["circles__px_projected_selection_shuffled"]

        # Accumulate results
        for key in output_dict.keys():
            per_sample_output[key].extend(output_dict[key])

    # Create DataFrame and explode nested lists
    calibration_batch_df = pd.DataFrame.from_dict(per_sample_output)

    explode_columns = [
        key for key, value in per_sample_output.items()
        if isinstance(value, list) and len(value) > 0
    ]

    calibration_batch_df = calibration_batch_df.explode(column=explode_columns)
    calibration_batch_df.set_index("image_id", inplace=True, drop=False)

    return calibration_batch_df

In [14]:
# ====================== FULL ITERATIVE CALIBRATION PIPELINE ======================

def calibrate_video_iterative(
    config: TVCalibConfig,
    ctx: Optional[TVCalibContext] = None,
    clean_temp: bool = True,
) -> pd.DataFrame:
    """
    Full iterative calibration pipeline for a video.

    Processing flow:
        1. Read video in batches to save memory.
        2. Save current batch of frames to temporary disk.
        3. Run TVCalib segmentation on the batch.
        4. Run TVCalib camera calibration.
        5. Clean up temporary files.
        6. Accumulate results in memory.

    Args:
        config: TVCalibConfig with all parameters
        ctx: Optional pre-initialized TVCalibContext
        clean_temp: Whether to clean temporary directory before starting

    Returns:
        pd.DataFrame: Final calibration results for all processed frames

    Raises:
        RuntimeError: If no frames were successfully calibrated
    """
    if ctx is None:
        ctx = TVCalibContext(config)

    # Prepare temporary directory for frame batches
    temp_root = prepare_temp_root(
        config.temp_root,
        clean=clean_temp,
    )

    calibration_batches: List[pd.DataFrame] = []

    # Iterate over video in batches
    video_batches = iter_video_frame_batches(
        video_path=config.video_path,
        batch_size=config.frame_batch_size,
        start_frame=0,
        max_frames=config.max_frames,
    )

    for batch_idx, (frame_indices, frames_bgr) in enumerate(
        tqdm(video_batches, desc="Processing video batches")
    ):
        batch_dir = temp_root / f"batch_{batch_idx:05d}"

        try:
            # Save batch to disk
            save_frame_batch_to_disk(
                frames_bgr=frames_bgr,
                frame_indices=frame_indices,
                batch_dir=batch_dir,
            )

            # Run segmentation
            image_ids, keypoints_raw = run_segmentation_on_image_dir(
                images_path=batch_dir,
                ctx=ctx,
            )

            # Run calibration
            calibration_batch_df = run_calibration_from_keypoints(
                image_ids=image_ids,
                keypoints_raw=keypoints_raw,
                ctx=ctx,
            )

            if len(calibration_batch_df) > 0:
                calibration_batches.append(calibration_batch_df)

        finally:
            # Always clean up temporary files
            if batch_dir.exists():
                shutil.rmtree(batch_dir)

            # Free GPU memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if len(calibration_batches) == 0:
        raise RuntimeError("Calibration produced no results.")

    # Combine all batches
    calibration_df = pd.concat(calibration_batches, axis=0)

    # Add frame_index for easier sorting and analysis
    calibration_df["frame_index"] = calibration_df["image_id"].apply(
        frame_index_from_image_id
    )
    calibration_df = calibration_df.sort_values("frame_index")
    calibration_df.set_index("image_id", inplace=True, drop=False)

    print(f"Calibration completed. Processed frames: {len(calibration_df)}")

    return calibration_df

In [15]:
# ====================== SEGMENTATION/CALIBRATION VISUALIZATION ======================

def visualize_calibration_frame(
    frame_bgr: np.ndarray,
    sample: pd.Series,
    ctx: TVCalibContext,
) -> np.ndarray:
    """
    Visualize TVCalib calibration result for a single frame.

    Args:
        frame_bgr: Input frame in OpenCV BGR format
        sample: Row from calibration DataFrame containing calibration data
        ctx: TVCalibContext with models and configuration

    Returns:
        np.ndarray: Annotated frame in BGR format (ready for OpenCV)
    """
    config = ctx.config

    # Convert BGR (OpenCV) → RGB → torch tensor
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Resize to the resolution used during calibration
    # (required for correct TVCalib visualization)
    frame_rgb = cv2.resize(
        frame_rgb,
        (config.image_width, config.image_height),
        interpolation=cv2.INTER_LINEAR,
    )

    image_tensor = T.functional.to_tensor(frame_rgb)

    # Extract camera parameters from calibration result
    cam = get_camera_from_per_sample_output(
        sample,
        config.lens_dist,
    )

    points_line = sample["points_line"]
    points_circle = sample["points_circle"]

    # Initialize matplotlib figure
    fig, ax = viz.init_figure(
        config.image_width,
        config.image_height,
    )

    # Draw original image
    ax = viz.draw_image(ax, image_tensor)

    # Draw reprojected field lines
    ax = viz.draw_reprojection(
        ax,
        ctx.object3dcpu,
        cam,
    )

    # Draw selected points (lines and circles)
    ax = viz.draw_selected_points(
        ax,
        ctx.object3dcpu,
        points_line,
        points_circle,
        kwargs_outer={
            "zorder": 1000,
            "rasterized": False,
            "s": 500,
            "alpha": 0.3,
            "facecolor": "none",
            "linewidths": 3,
        },
        kwargs_inner={
            "zorder": 1000,
            "rasterized": False,
            "s": 50,
            "marker": ".",
            "color": "k",
            "linewidths": 4.0,
        },
    )

    # Render matplotlib figure to numpy array
    fig.canvas.draw()
    annotated_rgb = np.asarray(fig.canvas.buffer_rgba())
    annotated_rgb = annotated_rgb[:, :, :3].copy()  # Remove alpha channel

    plt.close(fig)  # Clean up figure to prevent memory leaks

    # Convert back to BGR for OpenCV compatibility
    annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)

    return annotated_bgr

def visualize_calibration_frames_sequential(
    calibration_df: pd.DataFrame,
    video_path: str,
    ctx: TVCalibContext,
    max_frames: Optional[int] = None,
) -> List[np.ndarray]:
    """
    Efficient visualization of calibration results.

    Args:
        calibration_df: DataFrame with calibration results
        video_path: Path to the source video
        ctx: TVCalibContext with models and helpers
        max_frames: Maximum number of frames to visualize (for testing)

    Returns:
        List of annotated frames (in BGR format)
    """
    # Limit frames if requested (useful for testing)
    if max_frames is not None:
        df_vis = calibration_df.head(max_frames).copy()
    else:
        df_vis = calibration_df.copy()

    # Extract frame indices from image_id and sort
    df_vis["frame_index"] = df_vis["image_id"].apply(frame_index_from_image_id)
    df_vis = df_vis.sort_values("frame_index")

    needed_indices = set(df_vis["frame_index"].tolist())

    # Map frame_index → row for quick lookup
    sample_by_frame = {
        int(row.frame_index): row
        for _, row in df_vis.iterrows()
    }

    annotated_frames: List[np.ndarray] = []

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    max_needed_frame = max(needed_indices) if needed_indices else 0

    pbar = tqdm(
        total=df_vis.shape[0],
        desc="Visualizing calibration",
        smoothing=0.1, 
        mininterval=0.1,
    )

    current_idx = 0
    while current_idx <= max_needed_frame:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        if current_idx in needed_indices:
            sample = sample_by_frame[current_idx]
            annotated_frame = visualize_calibration_frame(
                frame_bgr=frame_bgr,
                sample=sample,
                ctx=ctx,
            )
            annotated_frames.append(annotated_frame)
            pbar.update(1)

        pbar.update(1)
        current_idx += 1

    pbar.close()
    cap.release()

    return annotated_frames

# Other modules

In [16]:
# ====================== PLAYER BALL ASSIGNER ======================

class PlayerBallAssigner:
    """
    Assigns the ball to the closest player based on foot proximity.
    """
    def __init__(self):
        self.max_player_ball_distance = 40  # pixels

    def assign_ball_to_player(
        self, 
        players: dict, 
        ball_bbox: list
    ) -> int:
        """
        Assign ball to the nearest player.

        Args:
            players: Dictionary of players in the current frame
            ball_bbox: Bounding box of the ball [x1, y1, x2, y2]

        Returns:
            int: player_id of the player who has the ball, or -1 if no player is close enough
        """
        if not players:
            return -1

        ball_position = get_center_of_bbox(ball_bbox)
        minimum_distance = 99999
        assigned_player = -1

        for player_id, player in players.items():
            player_bbox = player['bbox']

            # Measure distance to both bottom corners (feet area)
            distance_left = measure_distance(
                (player_bbox[0], player_bbox[3]), 
                ball_position
            )
            distance_right = measure_distance(
                (player_bbox[2], player_bbox[3]), 
                ball_position
            )

            distance = min(distance_left, distance_right)

            if distance < self.max_player_ball_distance and distance < minimum_distance:
                minimum_distance = distance
                assigned_player = player_id

        return assigned_player

In [31]:
# ====================== TRACKER ======================
class Tracker:
    """Class for tracking players, goalkeepers, referees, and the ball in a football video."""

    def __init__(
        self,
        model_path: str,
        # pitch_model_path: str,
        frame_size: Optional[tuple] = None,
        bytetrack_params: Optional[Dict[str, Any]] = None,
    ):
        """
        Args:
            model_path: path to the YOLO model file.
            frame_size: frame size (width, height), optional.
            bytetrack_params: parameters for sv.ByteTrack.
                If None or an empty dictionary is passed, reasonable default values are used.
        """
        self.model = YOLO(model_path)
        # self.pitch_model = YOLO(pitch_model_path)

        self.frame_size = frame_size
        self.class_names = self.model.names
        self.class_names_inverse = {v: k for k, v in self.class_names.items()}

        # Frequently used class IDs
        self.player_id = self.class_names_inverse.get("player")
        self.goalkeeper_id = self.class_names_inverse.get("goalkeeper")
        self.referee_id = self.class_names_inverse.get("referee")
        self.ball_id = self.class_names_inverse.get("ball")

        if bytetrack_params:
            self.tracker = sv.ByteTrack(**bytetrack_params)
        else:
            self.tracker = sv.ByteTrack()

    @torch.inference_mode()
    def _process_track_batch(
        self,
        batch: list,
        tracks: dict,
        start_frame: int,
        conf: float = 0.3,
        verbose: bool = True,
        quantize: bool = False,
        stream: bool = False,
    ):
        """
        Processes a batch of frames:
        detection -> conversion -> tracking -> result saving.

        Note: goalkeepers can be converted to the "player" class if needed.
        """
        detections_batch = self.model.predict(
            batch,
            conf=conf,
            verbose=verbose,
            quantize=quantize,
            stream=stream,
        )

        for i, detection in enumerate(detections_batch):
            cur_frame = start_frame + i

            # Convert detections to the supervision format
            sv_detections = sv.Detections.from_ultralytics(detection)

            # # Treat goalkeepers as players
            # if self.goalkeeper_id is not None and self.player_id is not None:
            #     mask = sv_detections.class_id == self.goalkeeper_id
            #     sv_detections.class_id[mask] = self.player_id
            #     sv_detections.data["class_name"][mask] = "player"

            # Run the tracker
            detections_with_tracks = self.tracker.update_with_detections(sv_detections)

            # Initialize empty dictionaries for the current frame
            tracks["players"].append({})
            tracks["goalkeepers"].append({})
            tracks["referees"].append({})
            tracks["ball"].append({})

            # Save players, goalkeepers, and referees with track_id
            for bbox, cls_id, trk_id in zip(
                detections_with_tracks.xyxy,
                detections_with_tracks.class_id,
                detections_with_tracks.tracker_id,
            ):
                if cls_id == self.player_id:
                    tracks["players"][cur_frame][int(trk_id)] = {"bbox": bbox}
                elif cls_id == self.goalkeeper_id:
                    tracks["goalkeepers"][cur_frame][int(trk_id)] = {"bbox": bbox}
                elif cls_id == self.referee_id:
                    tracks["referees"][cur_frame][int(trk_id)] = {"bbox": bbox}

            # The ball is usually not tracked by ByteTrack, so it is saved without track_id
            for j, (bbox, cls_id) in enumerate(
                zip(sv_detections.xyxy, sv_detections.class_id)
            ):
                if cls_id == self.ball_id:
                    tracks["ball"][cur_frame][j] = {"bbox": bbox}

    def get_object_tracks(
        self,
        frame_generator,
        batch_size: int = 20,
        total_frames: int = 0,
        conf: float = 0.3,
        quantize: bool = False,
        stream: bool = False,
        verbose: bool = True,
        stub_path: str = None,
        interpolate_ball: bool = True,
    ) -> dict:
        """
        Main method for obtaining tracks of all objects in the video.

        Supports:
            - caching results in a pickle stub file
            - batch processing to reduce memory usage
        """
        # Handle cached results (stub)
        if stub_path and os.path.exists(stub_path):
            if os.path.getsize(stub_path) > 0:
                try:
                    with open(stub_path, "rb") as f:
                        return pickle.load(f)
                except (EOFError, pickle.UnpicklingError):
                    print(f"File {stub_path} is corrupted -> recreating it")
                    os.remove(stub_path)
            else:
                os.remove(stub_path)

        tracks = {
            "players": [],
            "goalkeepers": [],
            "referees": [],
            "ball": [],
        }

        batch = []
        frame_num = 0

        # Progress bar (single smooth bar)
        with tqdm(
            total=total_frames,
            desc="Tracking",
            smoothing=0.1,
            mininterval=0.05,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
        ) as pbar:

            for frame in frame_generator:
                batch.append(frame)
                frame_num += 1

                if len(batch) == batch_size:
                    self._process_track_batch(
                        batch=batch,
                        tracks=tracks,
                        start_frame=frame_num - batch_size,
                        conf=conf,
                        verbose=verbose,
                        quantize=quantize,
                        stream=stream,
                    )
                    batch.clear()

                pbar.update(1)

            # Process remaining frames
            if batch:
                self._process_track_batch(
                    batch=batch,
                    tracks=tracks,
                    start_frame=frame_num - len(batch),
                    conf=conf,
                    verbose=verbose,
                    quantize=quantize,
                    stream=stream,
                )
                pbar.update(len(batch))

        if interpolate_ball:
            tracks["ball"] = interpolate_ball_positions(tracks["ball"])

        # Save cache if needed
        if stub_path:
            os.makedirs(os.path.dirname(stub_path), exist_ok=True)
            with open(stub_path, "wb") as f:
                pickle.dump(tracks, f)

        return tracks

In [18]:
# ====================== TEAM CLASSIFIER ======================
@dataclass
class TeamClassifierConfig:
    model_name: str = "google/siglip-base-patch16-224"
    batch_size: int = 32
    n_clusters: int = 2
    random_state: int = 0
    warmup_samples: int = 150  # Number of crops used for training
    buffer_frames: int = 10  # Number of frames collected before fixing an ID


class TeamClassifier:
    def __init__(
        self,
        config: Optional[TeamClassifierConfig] = None,
        device: Optional[str] = None,
    ):
        self.config = config or TeamClassifierConfig()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.processor = AutoProcessor.from_pretrained(self.config.model_name)
        self.model = SiglipVisionModel.from_pretrained(self.config.model_name).to(
            self.device
        ).eval()

        self.pca = PCA(n_components=3, random_state=self.config.random_state)
        self.kmeans = KMeans(
            n_clusters=self.config.n_clusters,
            n_init=20,
            random_state=self.config.random_state,
        )

        self.is_fitted = False
        self.team_colors: List[Tuple[int, int, int]] = []

        # Data accumulated before training
        self._warmup_crops: List[np.ndarray] = []

        # History for fixing IDs: {track_id: [list_of_team_ids]}
        self.track_history: Dict[int, List[int]] = {}

        # Final decision for each ID: {track_id: team_id}
        self.fixed_teams: Dict[int, int] = {}

        # List for storing references to player objects before training
        # Structure: [(player_dict_reference, track_id, crop), ...]
        self._pending_assignments = []

    def _get_vibrant_color(self, rgb: Tuple[int, int, int]) -> Tuple[int, int, int]:
        """Makes the color brighter and more saturated while preserving its hue."""
        r, g, b = [x / 255.0 for x in rgb]
        h, s, v = colorsys.rgb_to_hsv(r, g, b)

        # Set saturation and value close to maximum
        new_rgb = colorsys.hsv_to_rgb(h, 0.75, 0.9)

        return tuple(int(x * 255) for x in new_rgb)

    def fit(self, crops: List[np.ndarray]):
        """Fits the classifier on collected crops."""
        if not crops:
            return

        embeddings = self._extract_embeddings(crops)
        projections = self.pca.fit_transform(embeddings)
        labels = self.kmeans.fit_predict(projections)

        self.is_fitted = True

        raw_colors = self._compute_team_colors(crops, labels)
        self.team_colors = [self._get_vibrant_color(c) for c in raw_colors]

        # print(f"[INFO] TeamClassifier fitted. Colors: {self.team_colors}")

    def process_frame(self, frame, players, goalkeepers):
        if not self.is_fitted:
            for p_id, p_data in players.items():
                crop = self._crop_torso(frame, p_data["bbox"])

                if crop.size > 0:
                    self._warmup_crops.append(crop)

                    # Store a reference to the player dictionary to update it later
                    self._pending_assignments.append((p_data, p_id, crop))

            if len(self._warmup_crops) >= self.config.warmup_samples:
                self.fit(self._warmup_crops)

                # Fill missed frames retroactively
                self._apply_pending_assignments()
                self._warmup_crops.clear()

            return

        # If the classifier is already fitted, run the standard workflow
        self._classify_players(frame, players)
        self._classify_goalkeepers(players, goalkeepers)

    def _classify_players(self, frame: np.ndarray, player_dict: Dict[int, dict]):
        new_crops = []
        new_ids = []

        for t_id, player in player_dict.items():
            if t_id in self.fixed_teams:
                # If the ID is already fixed, assign a stable color
                team_id = self.fixed_teams[t_id]
                player["team_id"] = team_id
                player["team_color"] = self.team_colors[team_id]
            else:
                # Player is still in the statistics accumulation stage
                crop = self._crop_torso(frame, player["bbox"])

                if crop.size > 0:
                    new_crops.append(crop)
                    new_ids.append(t_id)

        if not new_crops:
            return

        # Predict team IDs for new players
        predicted_teams = self.predict(new_crops)

        for t_id, team_id in zip(new_ids, predicted_teams):
            if t_id not in self.track_history:
                self.track_history[t_id] = []

            self.track_history[t_id].append(int(team_id))

            smooth_team_id = Counter(self.track_history[t_id]).most_common(1)[0][0]

            player_dict[t_id]["team_id"] = smooth_team_id
            player_dict[t_id]["team_color"] = self.team_colors[smooth_team_id]

            if len(self.track_history[t_id]) >= self.config.buffer_frames:
                self.fixed_teams[t_id] = smooth_team_id

    def _classify_goalkeepers(
        self,
        players: Dict[int, dict],
        goalkeepers: Dict[int, dict],
    ):
        """Assigns a goalkeeper to the team whose players are closer to him."""
        if not goalkeepers or not players:
            return

        for g_id, goalie in goalkeepers.items():
            g_bbox = goalie["bbox"]
            g_center = np.array(
                [
                    (g_bbox[0] + g_bbox[2]) / 2,
                    (g_bbox[1] + g_bbox[3]) / 2,
                ]
            )

            distances = {0: [], 1: []}

            for p_id, p_data in players.items():
                if "team_id" in p_data:
                    p_bbox = p_data["bbox"]
                    p_center = np.array(
                        [
                            (p_bbox[0] + p_bbox[2]) / 2,
                            (p_bbox[1] + p_bbox[3]) / 2,
                        ]
                    )

                    dist = np.linalg.norm(g_center - p_center)
                    distances[p_data["team_id"]].append(dist)

            # Compute the average distance to the three closest players of each team
            avg_dist = {}

            for t_id in [0, 1]:
                if len(distances[t_id]) > 0:
                    avg_dist[t_id] = np.mean(sorted(distances[t_id])[:3])
                else:
                    avg_dist[t_id] = float("inf")

            assigned_team = min(avg_dist, key=avg_dist.get)

            goalie["team_id"] = assigned_team
            goalie["team_color"] = self.team_colors[assigned_team]

    def predict(self, crops: List[np.ndarray]) -> np.ndarray:
        embeddings = self._extract_embeddings(crops)
        projections = self.pca.transform(embeddings)

        return self.kmeans.predict(projections)

    def _extract_embeddings(self, crops: List[np.ndarray]) -> np.ndarray:
        from PIL import Image

        processed_images = [
            Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
            for c in crops
        ]

        all_embeddings = []

        for i in range(0, len(processed_images), self.config.batch_size):
            batch = processed_images[i : i + self.config.batch_size]
            inputs = self.processor(images=batch, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
                emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
                all_embeddings.append(emb)

        return normalize(np.concatenate(all_embeddings))

    def _compute_team_colors(
        self,
        crops: List[np.ndarray],
        labels: np.ndarray,
    ) -> List[Tuple[int, int, int]]:
        colors = []

        for team_id in range(self.config.n_clusters):
            team_crops = [
                c
                for c, lbl in zip(crops, labels)
                if lbl == team_id
            ]

            if not team_crops:
                colors.append((128, 128, 128))
                continue

            pixels = np.concatenate(
                [
                    cv2.resize(c, (16, 16)).reshape(-1, 3)
                    for c in team_crops
                ]
            )

            # Find the dominant color using a simple KMeans(1)
            km = KMeans(n_clusters=1, n_init=5).fit(pixels)
            colors.append(tuple(map(int, km.cluster_centers_[0])))

        return colors

    def _crop_torso(self, frame: np.ndarray, bbox: List[float]) -> np.ndarray:
        x1, y1, x2, y2 = map(int, bbox)
        h = y2 - y1

        # Use only the torso area, avoiding the pitch and shorts
        return frame[int(y1 + h * 0.15) : int(y1 + h * 0.5), x1:x2]

    def _apply_pending_assignments(self):
        """Fills team_id for players in frames that were used for training."""
        if not self._pending_assignments:
            return

        crops = [item[2] for item in self._pending_assignments]
        predicted_teams = self.predict(crops)

        for (p_data, p_id, _), team_id in zip(
            self._pending_assignments,
            predicted_teams,
        ):
            # Add the prediction to history so the ID fixing logic can work
            if p_id not in self.track_history:
                self.track_history[p_id] = []

            self.track_history[p_id].append(int(team_id))

            # Assign the current value to the frame dictionary
            p_data["team_id"] = int(team_id)
            p_data["team_color"] = self.team_colors[team_id]

        # Clear the queue, as it is no longer needed
        self._pending_assignments.clear()

In [19]:
# ====================== VIEW TRANSFORMER (HOMOGRAPHY) ======================

class ViewTransformer:
    """
    Transforms pixel coordinates to real-world meters using homography matrices.
    """

    def __init__(self, df_cam: pd.DataFrame):
        """
        Initialize with calibration DataFrame containing homography per frame.

        Expected image_id format: "frame_000123.png"
        """
        self.homographies: dict[int, np.ndarray] = {}

        for _, row in df_cam.iterrows():
            try:
                # Extract frame number from image_id
                image_id = str(row['image_id'])
                frame_num = int(image_id.split('_')[-1].split('.')[0])
                self.homographies[frame_num] = np.array(row['homography'])
            except (KeyError, ValueError, IndexError):
                continue

    def transform_to_meters(self, tracks: dict) -> dict:
        """
        Transform bounding box positions to real-world coordinates (meters)
        and store them in 'position_field' for each object.

        Args:
            tracks: Dictionary with tracking data

        Returns:
            Updated tracks dictionary with 'position_field' added
        """
        categories = ['players', 'goalkeepers', 'referees', 'ball']

        for cat in categories:
            if cat not in tracks:
                continue

            for frame_idx, frame_data in enumerate(tracks[cat]):
                # Get homography matrix for current frame
                h_matrix = self.homographies.get(frame_idx)
                if h_matrix is None:
                    continue

                for track_id, info in frame_data.items():
                    if 'bbox' not in info:
                        continue

                    x1, y1, x2, y2 = info['bbox']

                    # Anchor point: center for ball, bottom center (feet) for others
                    if cat == 'ball':
                        px, py = (x1 + x2) / 2, (y1 + y2) / 2
                    else:
                        px, py = (x1 + x2) / 2, y2

                    # Apply homography
                    point = np.array([px, py, 1.0])
                    world_point = h_matrix @ point

                    # Normalize by Z (perspective division)
                    if abs(world_point[2]) > 1e-8:
                        m_x = world_point[0] / world_point[2]
                        m_y = world_point[1] / world_point[2]

                        # Store world coordinates for kinematics, radar, etc.
                        tracks[cat][frame_idx][track_id]["position_field"] = (m_x, m_y)

        return tracks

In [20]:
# ====================== RADAR MINIMAP ======================

# Default constants

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

RADAR_W = 1050  # radar frame width in pixels
RADAR_H = 680  # radar frame height in pixels

# All colors are in BGR format
COLOR_GRASS = (56, 148, 50)  # green pitch
COLOR_LINES = (255, 255, 255)  # white pitch markings
COLOR_BALL = (255, 255, 255)  # ball
COLOR_REFEREE = (0, 165, 255)  # referee, orange
COLOR_OUTLINE = (0, 0, 0)  # circle outline
COLOR_DEFAULT = (180, 180, 180)  # player without an assigned team

R_PLAYER = 10  # field player circle radius
R_GOALKEEPER = 13  # goalkeeper circle radius
R_REFEREE = 8  # referee circle radius
R_BALL = 6  # ball circle radius


# Main class

class RadarMinimap:
    """
    Builds a 2D radar minimap and saves it as an .mp4 video.

    Parameters
    ----------
    template_path : str
        Path to template_pitch_t.png.
        Template format: white background with blue pitch marking lines.
    output_path : str
        Path to the output .mp4 file.
    fps : float
        Video FPS. Use fps from sv.VideoInfo.from_video_path().
    radar_w, radar_h : int
        Radar frame size. The aspect ratio should be close to 105:68.
    pitch_length, pitch_width : float
        Pitch dimensions in meters, used for coordinate scaling.
    show_speed : bool
        Whether to show speed in km/h next to the circle.
    show_id : bool
        Whether to show track_id next to the circle.
    """

    def __init__(
        self,
        template_path: str,
        output_path: str,
        fps: float,
        radar_w: int = RADAR_W,
        radar_h: int = RADAR_H,
        pitch_length: float = PITCH_LENGTH_M,
        pitch_width: float = PITCH_WIDTH_M,
        show_speed: bool = True,
        show_id: bool = False,
    ):
        self.output_path = str(output_path)
        self.fps = fps
        self.radar_w = radar_w
        self.radar_h = radar_h
        self.pitch_length = pitch_length
        self.pitch_width = pitch_width
        self.show_speed = show_speed
        self.show_id = show_id

        # Scaling factors: meters -> pixels
        self.sx = radar_w / pitch_length
        self.sy = radar_h / pitch_width

        # Build the reference empty pitch frame: background + pitch markings
        self._blank = self._build_blank_field(template_path)

        self._writer: Optional[cv2.VideoWriter] = None

    def build_from_tracks(self, tracks: dict) -> str:
        """
        Main method: iterates over all track frames and writes an mp4 file.

        tracks : dict
            {"players": [...], "goalkeepers": [...],
             "referees": [...], "ball": [...]}

            Each list represents video frames.
            Each frame is a dictionary:
              {track_id: {"position_field": (x_m, y_m),
                          "team_id": int,        # optional
                          "team_color": (R,G,B), # optional
                          "speed": float}}       # optional
        """
        n_frames = max(
            len(tracks.get("players", [])),
            len(tracks.get("goalkeepers", [])),
            len(tracks.get("referees", [])),
            len(tracks.get("ball", [])),
        )

        team_colors = _extract_team_colors(tracks)

        print(f"[RadarMinimap] Team colors (BGR): {team_colors}")
        print(f"[RadarMinimap] Rendering {n_frames} frames -> {self.output_path}")

        self.open_writer()

        for frame_idx in range(n_frames):
            frame = self.render_frame(tracks, frame_idx, team_colors)
            self.write_frame(frame)

            if frame_idx % 200 == 0:
                print(f"  {100 * frame_idx / n_frames:.1f}%  frame {frame_idx}/{n_frames}")

        self.close_writer()

        print(f"[RadarMinimap] Saved: {self.output_path}")

        return self.output_path

    def render_frame(
        self,
        tracks: dict,
        frame_idx: int,
        team_colors: Optional[dict] = None,
    ) -> np.ndarray:
        """
        Draws one radar frame. Can be called directly inside a custom loop.

        Returns a BGR ndarray with shape (H, W, 3) and dtype uint8.
        """
        if team_colors is None:
            team_colors = _extract_team_colors(tracks)

        canvas = self._blank.copy()

        players_d = _get_frame(tracks, "players", frame_idx)
        goalkeepers_d = _get_frame(tracks, "goalkeepers", frame_idx)
        referees_d = _get_frame(tracks, "referees", frame_idx)
        ball_d = _get_frame(tracks, "ball", frame_idx)

        # Drawing order: distant objects first, then the ball on top
        for tid, info in players_d.items():
            self._draw_dot(
                canvas,
                info,
                _team_color(info, team_colors),
                R_PLAYER,
                tid,
            )

        for tid, info in goalkeepers_d.items():
            self._draw_dot(
                canvas,
                info,
                _team_color(info, team_colors),
                R_GOALKEEPER,
                tid,
                outline_thickness=2,
            )

        for tid, info in referees_d.items():
            self._draw_dot(
                canvas,
                info,
                COLOR_REFEREE,
                R_REFEREE,
                tid,
            )

        for _, info in ball_d.items():
            self._draw_dot(
                canvas,
                info,
                COLOR_BALL,
                R_BALL,
                label=None,
                outline_color=(80, 80, 80),
            )

        return canvas

    def open_writer(self):
        """Opens VideoWriter. Required for frame-by-frame writing."""
        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        self._writer = cv2.VideoWriter(
            self.output_path,
            fourcc,
            self.fps,
            (self.radar_w, self.radar_h),
        )

        if not self._writer.isOpened():
            raise RuntimeError(
                f"VideoWriter failed to open: {self.output_path}\n"
                "Check the path and make sure the directory exists."
            )

    def write_frame(self, frame: np.ndarray):
        """Writes one BGR frame to the file."""
        assert self._writer is not None, "Call open_writer() first"
        self._writer.write(frame)

    def close_writer(self):
        """Finalizes the mp4 file."""
        if self._writer:
            self._writer.release()
            self._writer = None

    # Private methods

    def _build_blank_field(self, template_path: str) -> np.ndarray:
        """
        Creates an empty pitch: green background + white pitch markings.

        Algorithm:
          1. Read the template with a white background and blue lines.
          2. Extract blue pixels using BGR channel thresholds.
          3. Build a green canvas.
          4. Paint pixels white where the blue lines were detected.
        """
        tmpl = cv2.imread(str(template_path))

        if tmpl is None:
            raise FileNotFoundError(f"Template not found: {template_path}")

        tmpl = cv2.resize(
            tmpl,
            (self.radar_w, self.radar_h),
            interpolation=cv2.INTER_LINEAR,
        )

        # Extract blue lines: B is much greater than G and R
        b = tmpl[:, :, 0].astype(np.int16)
        g = tmpl[:, :, 1].astype(np.int16)
        r = tmpl[:, :, 2].astype(np.int16)

        line_mask = (
            (b - g > 55)
            & (b - r > 55)
            & (b > 80)
        ).astype(np.uint8) * 255

        # Slight morphological dilation to make lines thicker and remove gaps
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        line_mask = cv2.dilate(line_mask, k)

        # Green canvas
        blank = np.full(
            (self.radar_h, self.radar_w, 3),
            COLOR_GRASS,
            dtype=np.uint8,
        )

        # White lines
        blank[line_mask > 0] = COLOR_LINES

        return blank

    def _m_to_px(self, x_m: float, y_m: float) -> tuple:
        """
        Converts field coordinates in meters to radar frame pixels.

        Field coordinates:
        - (0, 0) is the pitch center;
        - x_m increases to the right;
        - y_m increases downward, following the image coordinate system.
        """
        px = int((x_m + self.pitch_length / 2) * self.sx)
        py = int((y_m + self.pitch_width / 2) * self.sy)

        px = max(0, min(self.radar_w - 1, px))
        py = max(0, min(self.radar_h - 1, py))

        return px, py

    def _draw_dot(
        self,
        canvas: np.ndarray,
        info: dict,
        color: tuple,
        radius: int,
        label=None,
        outline_thickness: int = 1,
        outline_color: tuple = COLOR_OUTLINE,
    ):
        """
        Draws one circle on the canvas.
        If position_field is missing, the object is skipped silently.
        """
        pos = info.get("position_field")

        if pos is None:
            return

        x_m, y_m = pos
        cx, cy = self._m_to_px(x_m, y_m)

        # Black outline for contrast on any background
        cv2.circle(
            canvas,
            (cx, cy),
            radius + outline_thickness,
            outline_color,
            -1,
        )

        # Colored fill
        cv2.circle(canvas, (cx, cy), radius, color, -1)

        # Text label: speed and/or track_id
        # parts = []
        # if self.show_id and label is not None:
        #     parts.append(str(label))
        # if self.show_speed and "speed" in info:
        #     parts.append(f"{info['speed']:.0f}")

        # if parts:
        #     text = " ".join(parts)
        #     tx, ty = cx + radius + 3, cy + 4

        #     # Text shadow
        #     cv2.putText(
        #         canvas,
        #         text,
        #         (tx + 1, ty + 1),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         0.38,
        #         (0, 0, 0),
        #         2,
        #         cv2.LINE_AA,
        #     )

        #     # Main text
        #     cv2.putText(
        #         canvas,
        #         text,
        #         (tx, ty),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         0.38,
        #         (255, 255, 255),
        #         1,
        #         cv2.LINE_AA,
        #     )


# Module-level helper functions

def _get_frame(tracks: dict, category: str, frame_idx: int) -> dict:
    """Safely returns frame data or an empty dictionary."""
    lst = tracks.get(category, [])

    return lst[frame_idx] if frame_idx < len(lst) else {}


def _team_color(info: dict, team_colors: dict) -> tuple:
    """Returns the player's team BGR color or the default color."""
    tid = info.get("team_id")

    if tid is not None and tid in team_colors:
        return team_colors[tid]

    return COLOR_DEFAULT


def _extract_team_colors(tracks: dict) -> dict:
    """
    Collects {team_id: color_bgr} from players and goalkeepers.

    Input:
        team_color is stored as (R, G, B)

    Output:
        color_bgr: (B, G, R), used by OpenCV
    """
    colors: dict = {}

    def process_frame(frame_data):
        for info in frame_data.values():
            tid = info.get("team_id")
            tc = info.get("team_color")

            if tid is None or tc is None or tid in colors:
                continue

            if isinstance(tc, (tuple, list)) and len(tc) == 3:
                try:
                    r, g, b = map(int, tc)
                    colors[tid] = (b, g, r)  # RGB -> BGR
                except (ValueError, TypeError):
                    continue

    # First, check players
    for frame_data in tracks.get("players", []):
        process_frame(frame_data)

        if len(colors) >= 2:
            return colors

    # Then, check goalkeepers if both teams were not found
    for frame_data in tracks.get("goalkeepers", []):
        process_frame(frame_data)

        if len(colors) >= 2:
            return colors

    return colors

In [21]:
# ====================== VORONOI MINIMAP ======================

class VoronoiMinimap(RadarMinimap):
    """
    Builds a separate top-down video with a Voronoi diagram.

    Logic:
      1. Takes player/goalkeeper coordinates from tracks[*][frame]["position_field"].
      2. Converts field coordinates to pixels using the same method as RadarMinimap.
      3. For each low-resolution pixel, finds the nearest player.
      4. Colors the nearest player's area using the color of his team.
      5. Draws players, referees, and the ball on top of the regions.
    """

    def __init__(
        self,
        template_path: str,
        output_path: str,
        fps: float,
        radar_w: int = RADAR_W,
        radar_h: int = RADAR_H,
        pitch_length: float = PITCH_LENGTH_M,
        pitch_width: float = PITCH_WIDTH_M,
        alpha: float = 0.38,
        grid_step: int = 4,
        draw_boundaries: bool = True,
        include_goalkeepers: bool = True,
        show_id: bool = False,
        show_speed: bool = False,
    ):
        super().__init__(
            template_path=template_path,
            output_path=output_path,
            fps=fps,
            radar_w=radar_w,
            radar_h=radar_h,
            pitch_length=pitch_length,
            pitch_width=pitch_width,
            show_speed=show_speed,
            show_id=show_id,
        )

        self.alpha = float(alpha)
        self.grid_step = max(1, int(grid_step))
        self.draw_boundaries = draw_boundaries
        self.include_goalkeepers = include_goalkeepers

    def build_from_tracks(self, tracks: dict) -> str:
        n_frames = max(
            len(tracks.get("players", [])),
            len(tracks.get("goalkeepers", [])),
            len(tracks.get("referees", [])),
            len(tracks.get("ball", [])),
        )

        if n_frames == 0:
            raise ValueError("tracks is empty: no frames available for video generation.")

        team_colors = _extract_team_colors(tracks)

        print(f"[VoronoiMinimap] Team colors BGR: {team_colors}")
        print(f"[VoronoiMinimap] Rendering {n_frames} frames -> {self.output_path}")

        self.open_writer()

        try:
            iterator = (
                tqdm(range(n_frames), desc="Voronoi video")
                if "tqdm" in globals()
                else range(n_frames)
            )

            for frame_idx in iterator:
                frame = self.render_frame(tracks, frame_idx, team_colors)
                self.write_frame(frame)
        finally:
            self.close_writer()

        print(f"[VoronoiMinimap] Saved: {self.output_path}")

        return self.output_path

    def render_frame(
        self,
        tracks: dict,
        frame_idx: int,
        team_colors: Optional[dict] = None,
    ) -> np.ndarray:
        if team_colors is None:
            team_colors = _extract_team_colors(tracks)

        canvas = self._blank.copy()

        control_objects = self._collect_control_objects(
            tracks,
            frame_idx,
            team_colors,
        )

        if len(control_objects) >= 2:
            canvas = self._draw_voronoi_regions(canvas, control_objects)

        self._draw_objects_on_top(canvas, tracks, frame_idx, team_colors)

        return canvas

    def _collect_control_objects(
        self,
        tracks: dict,
        frame_idx: int,
        team_colors: dict,
    ) -> list:
        """
        Collects players that participate in Voronoi region construction.
        Referees and the ball are not used for region construction.
        """
        categories = ["players"]

        if self.include_goalkeepers:
            categories.append("goalkeepers")

        objects = []

        for category in categories:
            for track_id, info in _get_frame(tracks, category, frame_idx).items():
                pos = info.get("position_field")

                if pos is None:
                    continue

                x_m, y_m = pos
                px, py = self._m_to_px(x_m, y_m)

                objects.append(
                    {
                        "track_id": track_id,
                        "category": category,
                        "info": info,
                        "px": px,
                        "py": py,
                        "color": _team_color(info, team_colors),
                    }
                )

        return objects

    def _draw_voronoi_regions(
        self,
        canvas: np.ndarray,
        objects: list,
    ) -> np.ndarray:
        """
        Quickly builds an approximate Voronoi diagram on a regular grid.
        grid_step=4 is usually accurate enough and much faster than full
        per-pixel calculation.
        """
        h, w = canvas.shape[:2]
        step = self.grid_step

        xs = np.arange(0, w, step, dtype=np.float32) + step / 2
        ys = np.arange(0, h, step, dtype=np.float32) + step / 2

        xs = np.clip(xs, 0, w - 1)
        ys = np.clip(ys, 0, h - 1)

        grid_x, grid_y = np.meshgrid(xs, ys)

        points = np.array(
            [[obj["px"], obj["py"]] for obj in objects],
            dtype=np.float32,
        )
        colors = np.array([obj["color"] for obj in objects], dtype=np.uint8)

        # labels[y, x] = index of the nearest player/goalkeeper
        dist2 = (
            (grid_x[..., None] - points[:, 0]) ** 2
            + (grid_y[..., None] - points[:, 1]) ** 2
        )
        labels = np.argmin(dist2, axis=2)

        voronoi_small = colors[labels]
        voronoi_full = cv2.resize(
            voronoi_small,
            (w, h),
            interpolation=cv2.INTER_NEAREST,
        )

        blended = cv2.addWeighted(
            canvas,
            1.0 - self.alpha,
            voronoi_full,
            self.alpha,
            0,
        )

        if self.draw_boundaries:
            edges = np.zeros(labels.shape, dtype=bool)
            edges[:, 1:] |= labels[:, 1:] != labels[:, :-1]
            edges[1:, :] |= labels[1:, :] != labels[:-1, :]

            edge_mask = cv2.resize(
                edges.astype(np.uint8) * 255,
                (w, h),
                interpolation=cv2.INTER_NEAREST,
            )
            edge_mask = cv2.dilate(
                edge_mask,
                np.ones((2, 2), np.uint8),
                iterations=1,
            )

            blended[edge_mask > 0] = (35, 35, 35)

        # Return white pitch markings on top of the regions to keep the field readable
        line_mask = np.all(canvas > 245, axis=2)
        blended[line_mask] = canvas[line_mask]

        return blended

    def _draw_objects_on_top(
        self,
        canvas: np.ndarray,
        tracks: dict,
        frame_idx: int,
        team_colors: dict,
    ):
        players_d = _get_frame(tracks, "players", frame_idx)
        goalkeepers_d = _get_frame(tracks, "goalkeepers", frame_idx)
        referees_d = _get_frame(tracks, "referees", frame_idx)
        ball_d = _get_frame(tracks, "ball", frame_idx)

        for track_id, info in players_d.items():
            self._draw_dot(
                canvas=canvas,
                info=info,
                color=_team_color(info, team_colors),
                radius=R_PLAYER,
                label=track_id,
            )

        for track_id, info in goalkeepers_d.items():
            self._draw_dot(
                canvas=canvas,
                info=info,
                color=_team_color(info, team_colors),
                radius=R_GOALKEEPER,
                label=track_id,
                outline_thickness=2,
            )

        for track_id, info in referees_d.items():
            self._draw_dot(
                canvas=canvas,
                info=info,
                color=COLOR_REFEREE,
                radius=R_REFEREE,
                label=track_id,
            )

        for _, info in ball_d.items():
            self._draw_dot(
                canvas=canvas,
                info=info,
                color=COLOR_BALL,
                radius=R_BALL,
                label=None,
                outline_color=(80, 80, 80),
            )


VORONOI_OUTPUT_PATH = "/kaggle/working/output/voronoi.mp4"

# Running

In [22]:
# ====================== RUN CALIBRATION ======================

# Configuration
config = TVCalibConfig(
    video_path="/kaggle/input/datasets/budunbudunov/some-football-videos/A1606b0e6_0 (3).mp4",
    temp_root="/kaggle/working/tvcalib_temp_batches",
    checkpoint="data/segment_localization/train_59.pt",
    image_width=1920,
    image_height=1080,
    
    # Test / development settings
    max_frames=750,                    # Set to None to process entire video
    
    # Performance settings
    frame_batch_size=128,
    batch_size_seg=8,
    batch_size_calib=64,
    nworkers=4,
    
    # Calibration settings
    optim_steps=3000,
    lens_dist=False,
    gpu=True,
)

# Initialize context and run full pipeline
ctx = TVCalibContext(config)

print("Starting video calibration...")
calibration_df = calibrate_video_iterative(
    config=config,
    ctx=ctx,
    clean_temp=True,
)

# Display results
print(f"\nCalibration completed successfully!")
print(f"Total frames processed: {len(calibration_df)}")
# print(f"DataFrame shape: {calibration_df.shape}")

# calibration_df.head()

TVCalib device: cuda
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 206MB/s]  


Starting video calibration...


Processing video batches: 0it [00:00, ?it/s]UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()


  0%|          | 0/3000 [00:00<?, ?it/s]

...

Processing video batches: 5it [09:00, 108.14s/it]UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()


  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

Processing video batches: 6it [10:43, 107.27s/it]

Calibration completed. Processed frames: 750

Calibration completed successfully!
Total frames processed: 750


In [23]:
# ====================== RUN CALIBRATION VISUALIZATION ======================

# Get original video FPS for correct playback speed
fps = get_video_fps(config.video_path)

print("Generating calibration visualization...")

calibration_annotated_frames = visualize_calibration_frames_sequential(
    calibration_df=calibration_df,
    video_path=config.video_path,
    ctx=ctx,
    max_frames=750,          # Set to None to visualize all frames
)

# Save annotated frames as video
output_calibration_video = save_frames_to_video(
    frames=calibration_annotated_frames,
    output_path="/kaggle/working/output/calibration_reprojection_preview.mp4",
    fps=fps,
)

print(f"✅ Visualization video successfully saved:")
print(f"   {output_calibration_video}")
print(f"   Total frames: {len(calibration_annotated_frames)} | FPS: {fps}")

Generating calibration visualization...


Visualizing calibration: 1500frame [04:38,  5.38frame/s]                     


✅ Visualization video successfully saved:
   /kaggle/working/output/calibration_reprojection_preview.mp4
   Total frames: 750 | FPS: 25.0


In [26]:
SOURCE_VIDEO_PATH = "/kaggle/input/datasets/budunbudunov/some-football-videos/A1606b0e6_0 (3).mp4"  # good one
MODEL_PATH = "/kaggle/input/models/budunbudunov/diploma-obj-detect/pytorch/default/1/best (2).pt"
PITCH_MODEL_PATH = "/kaggle/input/models/budunbudunov/pitch-points-model/pytorch/default/1/pitch_points_detect_model.pt"
STUB_PATH = "/kaggle/working/stubs/track_stubs.pkl"
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
conf=0.6
verbose=False
stream=True
batch_size=32
frame_size = (video_info.width, video_info.height)
fps = video_info.fps
total_frames = video_info.total_frames
bytetrack_params = {
    "track_activation_threshold": 0.65,      # или 0.55–0.7 — подбери экспериментально
    "lost_track_buffer": fps,                # даёт запас на пропуски ~1.5–2 секунды
    "minimum_matching_threshold": 0.9,     # стабильность трека мяча
    "frame_rate": fps,                       # твой реальный FPS
    "minimum_consecutive_frames": 10         # самый мощный фильтр против вспышек ложных мячей
}

In [32]:
def main():
    VIDEO_PATH = SOURCE_VIDEO_PATH

    # -------------------------
    # INIT
    # -------------------------
    print("Initialization...")

    tracker = Tracker(
        model_path=MODEL_PATH,
        frame_size=frame_size,
        bytetrack_params=bytetrack_params,
    )
    team_classifier = TeamClassifier()

    # -------------------------
    # 1. Tracking
    # -------------------------
    print("Running tracking...")
    tracks = tracker.get_object_tracks(
        frame_generator=sv.get_video_frames_generator(VIDEO_PATH),
        stub_path=STUB_PATH,
        batch_size=batch_size,
        total_frames=total_frames,
        conf=conf,
        verbose=verbose,
        stream=stream,
    )

    # -------------------------
    # 2. Team assignment and ball control
    # -------------------------
    print("Assigning teams...")
    frame_gen = load_video_frames(VIDEO_PATH)
    team_ball_control = process_tracks(
        frame_gen,
        tracks,
        team_classifier,
        ball_control=True,
    )

    # -------------------------
    # 3. Pixel -> world coordinates: perspective transform
    # -------------------------
    print("Transforming coordinates to field plane...")

    # Assume that the `df` variable, a DataFrame with homography data,
    # has already been created in the notebook.
    # If not, run the "get df_calib" cell first.
    if "calibration_df" not in globals():
        raise RuntimeError(
            "DataFrame 'df' was not found. Run the tvcalib calibration cell first."
        )

    # Initialize the transformer using df_cam with homography data
    view_transformer = ViewTransformer(calibration_df)

    # Add 'position_field'
    tracks = view_transformer.transform_to_meters(tracks)

    # Add 'speed'
    tracks = calculate_kinematics(tracks, fps)

    radar = RadarMinimap(
        template_path="/kaggle/working/tvcalib/template_pitch_t.png",
        output_path="/kaggle/working/output/radar.mp4",
        fps=fps,
    )
    radar.build_from_tracks(tracks)

    voronoi = VoronoiMinimap(
        template_path="/kaggle/working/tvcalib/template_pitch_t.png",
        output_path=VORONOI_OUTPUT_PATH,
        fps=fps,
        alpha=0.38,  # region fill transparency
        grid_step=4,  # lower = more accurate but slower; 3-5 is usually acceptable
        draw_boundaries=True,
        include_goalkeepers=True,
        show_id=False,
        show_speed=False,
    )
    voronoi.build_from_tracks(tracks)

    # -------------------------
    # 5. Render annotated video with speed overlay
    # -------------------------
    print("Rendering main video with speed...")
    frame_gen = load_video_frames(VIDEO_PATH)
    annotated_frames = render_video(frame_gen, tracks, team_ball_control)

    # Add speed visualization
    annotated_frames_with_speed = draw_speed(annotated_frames, tracks)

    save_video(
        annotated_frames_with_speed,
        "/kaggle/working/output/video/annotated_with_speed3.mp4",
        fps,
    )

    print("Cleaning temporary files...")
    frames_dir = Path("/kaggle/working/frames")

    if frames_dir.exists():
        shutil.rmtree(frames_dir)
        print(f"Removed directory: {frames_dir}")

    print("Done! Saved: main video and radar video.")


    return tracks, calibration_df, team_ball_control

In [33]:
tracks, df, team_ball_control = main()

Initialization...


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.final_layer_norm.bias                             | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.b

Running tracking...
Assigning teams...
Transforming coordinates to field plane...
[RadarMinimap] Team colors (BGR): {1: (229, 179, 57), 0: (195, 229, 57)}
[RadarMinimap] Rendering 750 frames -> /kaggle/working/output/radar.mp4
  0.0%  frame 0/750
  26.7%  frame 200/750
  53.3%  frame 400/750
  80.0%  frame 600/750
[RadarMinimap] Saved: /kaggle/working/output/radar.mp4
[VoronoiMinimap] Team colors BGR: {1: (229, 179, 57), 0: (195, 229, 57)}
[VoronoiMinimap] Rendering 750 frames -> /kaggle/working/output/voronoi.mp4


Voronoi video: 100%|██████████| 750/750 [00:24<00:00, 31.18it/s]


[VoronoiMinimap] Saved: /kaggle/working/output/voronoi.mp4
Rendering main video with speed...
Created directory: /kaggle/working/output/video
Video created: /kaggle/working/output/video/annotated_with_speed3.mp4
Resolution: 1920x1080, FPS: 25
Video successfully saved: /kaggle/working/output/video/annotated_with_speed3.mp4 (750 frames)
Cleaning temporary files...
Done! Saved: main video and radar video.
